# <span style="color:#c8b6ff">**Deciding with Numbers / Décider avec des chiffres**</span>

**Linear Regression: from raw data to weighted decisions**

> *Part of the **AI From Scratch** series - a progressive introduction to artificial intelligence for everyone.*

# <span style="color:#c8b6ff">**1. Welcome & Objectives**</span>

## Ce que tu vas apprendre ici

Ce notebook est le point de départ de toute la série. Pas de prérequis. On part de zéro (des tableaux de données) et on arrive à construire un modèle de régression linéaire qui fait des prédictions réelles.

À la fin de ce notebook, tu sauras :

- lire et comprendre un tableau de données
- distinguer les types de variables
- calculer des statistiques de base et comprendre ce qu'elles disent
- visualiser des relations entre variables
- comprendre ce qu'est un poids et pourquoi ça compte
- construire une régression linéaire à la main, puis avec scikit-learn
- appliquer tout ça sur un exemple concret : **les retards de vols**

### Comment utiliser ce notebook?

- **Lecture seule** : tu peux lire les cellules markdown et regarder les outputs sans rien exécuter
- **Run it yourself** : exécute les cellules dans l'ordre avec `Shift + Enter`
- **Modifie les valeurs** : les cellules marquées `# ✏️ ESSAIE DE MODIFIER` t'invitent à jouer avec les paramètres


# <span style="color:#c8b6ff">**2. The Intuition First: L'intuition avant tout**</span>

## Une **décision**, c'est quoi ?

Imagine que tu dois décider si tu prends ton parapluie ce matin. Tu regardes plusieurs choses :

- il y a des nuages → signal fort
- la météo annonce 80% de pluie → signal fort
- c'est la saison des pluies → signal moyen
- hier il faisait beau → signal faible

Tu **agrèges** ces signaux, tu leur donnes un poids mental, et tu décides.

**C'est exactement ce que fait une régression linéaire.**

Elle prend plusieurs informations (on les appelle des **features** ou **variables**), leur attribue un poids, les additionne, et produit une valeur de sortie.

$$y = w_0 + w_1 x_1 + w_2 x_2 + \dots + w_n x_n$$

- $x_1, x_2, \dots$ → les informations d'entrée
- $w_1, w_2, \dots$ → les poids (ce que la machine apprend)
- $y$ → la valeur prédite
- $w_0$ → le biais (une valeur de base, même quand tout est à zéro)

## Vocabulaire clé

| Terme | Définition simple |
|-------|------------------|
| Feature / variable | Une information d'entrée (ex: température, heure) |
| Target / cible | Ce qu'on veut prédire (ex: retard en minutes) |
| Poids (weight) | L'importance donnée à chaque feature |
| Modèle | L'équation apprise à partir des données |
| Entraînement | Le processus d'apprentissage des poids |
| Prédiction | La valeur que le modèle produit pour un nouvel exemple |


# <span style="color:#c8b6ff">**3. The Math - Step by Step**</span>

## 3.1. Tableaux et données tabulaires

Toute l'IA commence par un tableau. Chaque **ligne** est un exemple. Chaque **colonne** est une information.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (10, 5)
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

# Un tableau simple - 8 vols
data = {
    'retard_depart_min': [10, 45, 0, 120, 5, 80, 15, 60],
    'meteo_score':       [2,  8,  1,  9,  1,  7,  3,  6],   # 1=beau, 10=tempête
    'heure_depart':      [8, 18,  6, 21,  7, 17,  9, 20],   # heure du vol
    'nb_escales':        [0,  1,  0,  2,  0,  1,  0,  1],   # nombre d'escales
    'retard_arrivee_min':[12, 55,  2, 140,  8, 95, 18, 72]  # notre CIBLE/TARGET
}

df = pd.DataFrame(data)
print('Notre tableau de données :')
df

**Ce qu'on lit dans ce tableau :**
- Chaque ligne = un vol
- Les colonnes de gauche = nos **features** (informations d'entrée)
- La dernière colonne = notre **target** (ce qu'on veut prédire)

La question qu'on pose à la machine : **<span style="color:#00b4d8">à partir du retard au départ, de la météo, de l'heure et du nombre d'escales - peut-on prédire le retard à l'arrivée ?</span>**

## 3.2. Variables et types

Toutes les colonnes ne se traitent de la même façon. Il y a deux grandes familles :

In [ ]:
# Variables numériques → des nombres avec lesquels on peut calculer
print('Variables numériques :')
print(df[['retard_depart_min', 'meteo_score', 'heure_depart']].dtypes)

print()

# Variable catégorielle → des catégories (ex: type de compagnie)
compagnies = ['Air France', 'EasyJet', 'Air France', 'Ryanair',
              'Air France', 'EasyJet', 'Ryanair', 'Air France']
df['compagnie'] = compagnies
print('Variable catégorielle :')
print(df['compagnie'].value_counts())

**Pourquoi ça compte ?**

- Une variable **numérique** : on peut directement la multiplier par un poids
- Une variable **catégorielle** : il faut d'abord la transformer en nombres (on appelle ça l'encodage)

Pour l'instant on travaille avec les numériques (c'est plus simple pour commencer).

## 3.3. Moyenne, médiane, écart-type

Avant de modéliser quoi que ce soit, on regarde les données. Trois indicateurs suffisent pour commencer :

In [ ]:
retards = df['retard_arrivee_min']

print(f'Moyenne   : {retards.mean():.1f} min  → le retard "typique"')
print(f'Médiane   : {retards.median():.1f} min  → la valeur du milieu (moins sensible aux extrêmes)')
print(f'Écart-type: {retards.std():.1f} min  → à quel point les retards varient')
print(f'Min / Max : {retards.min()} / {retards.max()} min')

# Visualisation
fig, ax = plt.subplots()
ax.bar(range(len(retards)), retards, color='#0077b6', alpha=0.8)
ax.axhline(retards.mean(), color='#E74C3C', linestyle='--', linewidth=1.5, label=f'Moyenne : {retards.mean():.0f} min')
ax.axhline(retards.median(), color='#2ECC71', linestyle='--', linewidth=1.5, label=f'Médiane : {retards.median():.0f} min')
ax.set_xlabel('Vol')
ax.set_ylabel('Retard à l\'arrivée (min)')
ax.set_title('Retards à l\'arrivée - vue d\'ensemble')
ax.legend()
ax.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.show()

**Ce qu'on retient :**
- La **moyenne** est sensible aux valeurs extrêmes - un vol avec 4h de retard tire la moyenne vers le haut
- La **médiane** est plus robuste - elle dit juste "la moitié des vols sont en dessous de cette valeur"
- L'**écart-type** mesure la dispersion - un écart-type élevé = les retards sont très variables

## 3.4. Corrélation et visualisation

La corrélation mesure à quel point deux variables bougent ensemble.

- **+1** : quand l'une monte, l'autre monte aussi (parfaitement)
- **-1** : quand l'une monte, l'autre descend (parfaitement)
- **0** : aucune relation

C'est la première question à poser : **<span style="color:#00b4d8">mes features ont-elles un lien avec ma target ?</span>**

In [ ]:
features = ['retard_depart_min', 'meteo_score', 'heure_depart', 'nb_escales']

print('Corrélation avec le retard à l\'arrivée :')
correlations = df[features + ['retard_arrivee_min']].corr()['retard_arrivee_min'].drop('retard_arrivee_min')
print(correlations.sort_values(ascending=False).round(2))

# Scatter plots
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
colors = ['#4A90D9', '#E74C3C', '#2ECC71', '#F39C12']

for i, (feat, color) in enumerate(zip(features, colors)):
    axes[i].scatter(df[feat], df['retard_arrivee_min'], color=color, alpha=0.8, s=80)
    axes[i].set_xlabel(feat.replace('_', ' '))
    axes[i].set_ylabel('Retard arrivée (min)' if i == 0 else '')
    corr_val = correlations[feat]
    axes[i].set_title(f'corr = {corr_val:.2f}')
    axes[i].spines['top'].set_visible(False)
    axes[i].spines['right'].set_visible(False)

plt.suptitle('Relation entre chaque feature et le retard à l\'arrivée', y=1.02)
plt.tight_layout()
plt.show()

**Ce qu'on observe :**
- Le retard au départ a une forte corrélation avec le retard à l'arrivée → **feature très utile**
- La météo aussi, logique
- L'heure de départ a peu d'effet ici, dans **NOTRE** cas (peut être oui dans s'autres contextes)

> Une bonne feature, c'est une variable qui **varie avec la target**. Si la corrélation est proche de 0, la feature n'apporte pas grand chose au modèle.

## 3.5. La notion de poids

Maintenant qu'on sait quelles features comptent, on peut introduire l'idée de **poids**.

Un poids, c'est simplement : **<span style="color:#00b4d8">combien cette feature influence la prédiction ?</span>**

In [ ]:
from ipywidgets import interact, FloatSlider, IntSlider

def predict(w0, w1, w2, w3, w4, x1, x2, x3, x4):
    prediction = w0 + w1*x1 + w2*x2 + w3*x3 + w4*x4
    
    print("Calcul détaillé :")
    print(f"{'Biais :':20} {w0:>10.1f}")
    print(f"{'Retard départ (' + str(x1) + ' min) :':20} {w1*x1:>10.1f}")
    print(f"{'Météo score (' + str(x2) + ') :':20} {w2*x2:>10.1f}")
    print(f"{'Heure départ (' + str(x3) + 'h) :':20} {w3*x3:>10.1f}")
    print(f"{'Nb escales (' + str(x4) + ') :':20} {w4*x4:>10.1f}")
    print("-" * 40)
    print(f"{'Retard prédit :':20} {prediction:>10.1f} min")

interact(
    predict,
    w0=FloatSlider(value=2.0, min=-10, max=10, step=0.1),
    w1=FloatSlider(value=0.9, min=-2, max=2, step=0.1),
    w2=FloatSlider(value=3.5, min=-5, max=5, step=0.1),
    w3=FloatSlider(value=0.2, min=-1, max=1, step=0.1),
    w4=FloatSlider(value=8.0, min=0, max=20, step=0.5),
    x1=IntSlider(value=30, min=0, max=300, step=5),
    x2=IntSlider(value=6, min=0, max=10, step=1),
    x3=IntSlider(value=17, min=0, max=23, step=1),
    x4=IntSlider(value=1, min=0, max=3, step=1),
)

In [ ]:
# ✏️ ESSAIE DE MODIFIER ces poids et observe l'effet sur la prédiction

w0 = 2.0   # biais — valeur de base
w1 = 0.9   # poids du retard au départ
w2 = 3.5   # poids de la météo
w3 = 0.2   # poids de l'heure
w4 = 8.0   # poids du nombre d'escales

# Un vol : 30 min de retard au départ, météo score 6, départ à 17h, 1 escale
x1, x2, x3, x4 = 30, 6, 17, 1

prediction = w0 + w1*x1 + w2*x2 + w3*x3 + w4*x4

print("Calcul détaillé :")
print(f"{'Biais :':20} {w0:>10.1f}")
print(f"{'Retard départ (' + str(x1) + ' min) :':20} {w1*x1:>10.1f}")
print(f"{'Météo score (' + str(x2) + ') :':20} {w2*x2:>10.1f}")
print(f"{'Heure départ (' + str(x3) + 'h) :':20} {w3*x3:>10.1f}")
print(f"{'Nb escales (' + str(x4) + ') :':20} {w4*x4:>10.1f}")
print("-" * 40)
print(f"{'Retard prédit :':20} {prediction:>10.1f} min")

In [ ]:
# <span style="color:#c8b6ff">...</span>
# <span style="color:#00b4d8">